# 01. Data Preparation

This notebook transforms the raw Kaggle *Corporación Favorita Grocery Sales Forecasting* dataset into three clean, analysis-ready tables — `sales.csv`, `products.csv`, and `inventory.csv` — that feed every downstream layer of the Retail Decision AI pipeline. It streams and filters the large `train.csv` file down to a manageable subset, standardises product metadata, and synthesises realistic inventory state that is absent from the original dataset.

---

**Design Decisions**

- **10 stores × 100 items:** Keeps the working dataset at a scale that is comfortable for a portfolio project (≈400 K rows) while preserving real sales patterns, seasonality, and store-level variance from the original data.
- **Start date 2016-01-01:** Provides full-year coverage for 2016 and a partial year for 2017, giving the metrics engine enough history to compute meaningful trends without loading years of earlier data.
- **`np.random` seed = 42:** All synthetic columns are generated with a fixed seed so that every notebook run produces identical inventory values, making results fully reproducible across environments.

## **Import Libraries**

In [ ]:
import pandas as pd
from pathlib import Path
import numpy as np


## **Load Raw Data**

We load `train.csv` (sales transactions), `items.csv` (product metadata), and `stores.csv` (store attributes) — the three raw files from the Kaggle dataset that contain everything needed to build the processed tables.

In [ ]:
RAW_DIR = Path("../data/raw")

train_path =  RAW_DIR / "train.csv"
items_path =  RAW_DIR / "items.csv"
stores_path =  RAW_DIR / "stores.csv"

print("train exist: ", train_path.exists())
print("items exist: ", items_path.exists())
print("stores exist: ", stores_path.exists())

In [ ]:
items = pd.read_csv(items_path)
stores = pd.read_csv(stores_path)

print("Items shape: ", items.shape)
print("Stores shape: ", stores.shape)

display(items.head(3))
display(stores.head(3))

In [ ]:
selected_stores = stores["store_nbr"].head(10).tolist()
selected_items = items["item_nbr"].head(100).tolist()

print("Selected stores:", selected_stores)
print("Number of selected stores:", len(selected_stores))
print("Selected items:", selected_items)
print("Numbers of selected items:", len(selected_items))

## **Subset Sales**

`train.csv` contains over 125 million rows and cannot be loaded into memory at once. We use pandas `read_csv` with `chunksize=500_000` to stream the file in batches, applying date, store, and item filters per chunk before concatenating only the matching rows — keeping peak memory usage low regardless of the full file size.

In [ ]:
chunk_size = 500_000
filtered_chunks = []

usecols = ["date", "store_nbr", "item_nbr", "unit_sales", "onpromotion"]

for chunk in pd.read_csv(train_path, usecols=usecols, chunksize=chunk_size):
    chunk["date"] = pd.to_datetime(chunk["date"])

    filtered = chunk[
        (chunk["date"] >= "2016-01-01") &
        (chunk["store_nbr"].isin(selected_stores)) &
        (chunk["item_nbr"].isin(selected_items))
    ].copy()

    filtered_chunks.append(filtered)

    print("Filtered rows collected so far:", sum(len(c) for c in filtered_chunks))

In [ ]:
sales_subset = pd.concat(filtered_chunks, ignore_index=True)

print("Shape of sales_subset: ", sales_subset.shape)
display(sales_subset.head(5))
display(sales_subset.tail(5))

## **Clean Sales**

### Negative Clipping

The Kaggle dataset records product returns as negative `unit_sales` values; since our pipeline models forward-looking demand rather than return flows, we clip these to zero rather than dropping the rows.

In [ ]:
negative_count = (sales_subset["unit_sales"] < 0).sum()
print("Negative unit_sales rows before cleaning:", negative_count)

sales_subset["unit_sales"] = sales_subset["unit_sales"].clip(lower=0)

negative_count_after = (sales_subset["unit_sales"] < 0).sum()
print("Negative unit_sales rows after cleaning:", negative_count_after)

In [ ]:
display(
    sales_subset[sales_subset["unit_sales"] < 0]
    .sort_values("unit_sales")
    .head(10)
)

In [ ]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

sales_path = processed_dir / "sales.csv"

sales_subset.to_csv(sales_path, index=False)

print("Saved sales subset to:", sales_path)

In [ ]:
print("Dataset info:")
print(sales_subset.info())

print("\nUnique stores:", sales_subset["store_nbr"].nunique())
print("Unique items:", sales_subset["item_nbr"].nunique())

print("\nDate range:")
print(sales_subset["date"].min(), "→", sales_subset["date"].max())

## **Build Products Table**

The products table is a dimension table that maps each `product_id` to its human-readable name, category (product family), class code, and perishability flag — providing the context layer that makes inventory and sales metrics interpretable.

### Inspect `items.csv`

In [ ]:
items = pd.read_csv("../data/raw/items.csv")

print("Shape:", items.shape)
display(items.head())

print("\nColumns:")
print(items.columns)

### Build `products.csv`

In [ ]:
products = items.copy()

products = products.rename(columns={
    "item_nbr": "product_id",
    "family": "category",
    "class": "product_class"

})

products["product_name"] = "Product " + products["product_id"].astype(str)

products = products[
    [
        "product_id",
        "product_name",
        "category",
        "product_class",
        "perishable"
    ]
]

print("Products shape:", products.shape)
display(products.head())
display(products.tail())

In [ ]:
products.perishable.value_counts()

### Export `products.csv`

In [ ]:
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

products_path = processed_dir / "products.csv"

products.to_csv(products_path, index=False)

print("Saved products to:", products_path)

## **Build Synthetic Inventory**

The Kaggle dataset does not include stock-level data, so inventory state (stock levels, reorder points, lead times, stockout frequency) is synthetically generated from the observed sales distributions. All random draws use `np.random` with `seed=42`, ensuring that the generated inventory is deterministic and fully reproducible across runs.

### Calculate Average Daily Sales

`avg_daily_sales` per store–product pair is the single most important derived signal in this notebook: it drives stock level sizing, reorder point calculation, and is the baseline demand estimate used by the scoring and metrics engines in later pipeline layers.

In [ ]:
sales = pd.read_csv("../data/processed/sales.csv", parse_dates=["date"])

avg_sales = (
    sales
    .groupby(["store_nbr", "item_nbr"])["unit_sales"]
    .mean()
    .reset_index()
)

avg_sales = avg_sales.rename(columns={
    "store_nbr": "store_id",
    "item_nbr": "product_id",
    "unit_sales": "avg_daily_sales"
})

print("Shape:", avg_sales.shape)
display(avg_sales.head())

In [ ]:
stores = sales["store_nbr"].unique()
products = sales["item_nbr"].unique()

full_grid = pd.MultiIndex.from_product(
    [stores, products],
    names=["store_id", "product_id"]
).to_frame(index=False)

print("Full grid shape:", full_grid.shape)
display(full_grid.head())

In [ ]:
inventory_base = full_grid.merge(
    avg_sales,
    on=["store_id", "product_id"],
    how="left"
)

inventory_base["avg_daily_sales"] = inventory_base["avg_daily_sales"].fillna(0)

print("Shape:", inventory_base.shape)
display(inventory_base.head())

In [ ]:
np.random.seed(42)

inventory = inventory_base.copy()

# 1) lead time supplier
inventory["supplier_lead_time_days"] = np.random.randint(2, 8, size=len(inventory))

# 2) order fulfillment time
inventory["order_fulfillment_time_days"] = np.random.randint(1, 4, size=len(inventory))

# 3) coverage days for stock
coverage_days = np.random.randint(7, 22, size=len(inventory))

# 4) stock level based on demand
inventory["stock_level"] = np.ceil(
    inventory["avg_daily_sales"] * coverage_days
).astype(int)

# optional: for zero-demand items, give small base stock
inventory.loc[inventory["avg_daily_sales"] == 0, "stock_level"] = np.random.randint(
    0, 6, size=(inventory["avg_daily_sales"] == 0).sum()
)

# 5) reorder point
inventory["reorder_point"] = np.ceil(
    inventory["avg_daily_sales"] * inventory["supplier_lead_time_days"]
).astype(int)

# 6) stockout frequency
inventory["stockout_frequency"] = np.round(
    np.random.uniform(0.00, 0.20, size=len(inventory)),
    3
)

inventory["reorder_point"] = inventory["reorder_point"].clip(lower=1)

display(inventory.head())
print("Shape:", inventory.shape)

In [ ]:
inventory_final = inventory.copy()

inventory_final = inventory_final[
    [
        "product_id",
        "store_id",
        "stock_level",
        "supplier_lead_time_days",
        "reorder_point",
        "stockout_frequency",
        "order_fulfillment_time_days"
    ]
]

processed_dir = Path("../data/processed")
inventory_path = processed_dir / "inventory.csv"

inventory_final.to_csv(inventory_path, index=False)

print("Saved inventory to:", inventory_path)